#  DeepFire Forecaster v2
### Hybrid CNN-Transformer for Multi-Day Wildfire Spread Prediction
**Dataset:** TS-SAT Fire (Kaggle)  
**Task:** Given 3 days of VIIRS thermal + weather data → predict fire probability heatmaps for **Days 4, 5, and 6**  
**Architecture:** Spatial CNN Encoder → Spatiotemporal Transformer → Temporal Attention Pooling → Multi-Day UNet Decoder  
**Loss:** Masked Focal-Dice Loss (handles severe class imbalance)  
**Changes v2:**
-  Multi-day prediction: outputs 3 future days simultaneously
-  Weather features from Day 3 (current day) fused into decoder
-  ESRI_LULC folders fully excluded from inputs
-  T4×2 dual-GPU support with DataParallel + AMP
-  Parallel preprocessing with multiprocessing
-  Evaluation: 3 best / 3 worst / 3 random samples (no subfolders)

---

In [ ]:
import subprocess, sys

required = ["rasterio", "tqdm"]
for pkg in required:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

print(" All dependencies ready.")


## 1. Imports & Reproducibility

In [ ]:
import os, glob, json, math, warnings, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import rasterio
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
from matplotlib.colors import ListedColormap
from torch.utils.data import Dataset, DataLoader, random_split
from tqdm.auto import tqdm
warnings.filterwarnings("ignore")

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42

def set_seed(seed: int = SEED):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False   # set True for speed if input sizes fixed

set_seed()

# ── Device: T4×2 support ──────────────────────────────────────────────────────
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
N_GPUS      = torch.cuda.device_count()
USE_AMP     = torch.cuda.is_available()          # Automatic Mixed Precision
print(f" Device      : {DEVICE}")
print(f" GPUs found  : {N_GPUS}  ({'DataParallel enabled' if N_GPUS > 1 else 'single GPU'})")
print(f" AMP         : {USE_AMP}")
print(f" PyTorch     : {torch.__version__}")


## 2. Global Configuration

In [ ]:
# ─── Paths ───────────────────────────────────────────────────────────────────
DATA_ROOT  = "/kaggle/input/ts-satfire/ts-satfire"
SAVE_DIR   = "/kaggle/working"
MODEL_PATH = os.path.join(SAVE_DIR, "deepfire_best.pt")

# ─── Data ────────────────────────────────────────────────────────────────────
SPATIAL_SIZE   = 128          # Resize all imagery to S×S pixels
SEQ_LEN        = 3            # Number of input days (Days 1–3)
PRED_HORIZON   = 3            # Predict next N days simultaneously (Days 4–6)
VIIRS_CHANNELS = 8            # Bands per VIIRS_Day .tif
NIGHT_CHANNELS = 2            # Bands per VIIRS_Night .tif
USE_NIGHT      = True         # Fuse VIIRS_Night observations
VIIRS_CLIP_MIN = 200.0        # Brightness temperature physical range (K)
VIIRS_CLIP_MAX = 400.0

# ─── Weather channels (ERA5-style bands from VIIRS_Day day SEQ_LEN) ──────────
# We treat the last SEQ_LEN VIIRS day frame as a proxy "current weather" signal.
# These are the same 8+2 bands, re-used to condition the decoder.
WEATHER_DIM = 16              # Projected weather feature dimension

# ─── Architecture ────────────────────────────────────────────────────────────
EMBED_DIM  = 128
NUM_HEADS  = 4
NUM_LAYERS = 3
DROPOUT    = 0.1

# ─── Training ────────────────────────────────────────────────────────────────
# With T4×2 + AMP we can afford a larger effective batch.
BATCH_SIZE   = 4              # Per-GPU; effective = BATCH_SIZE * N_GPUS
EPOCHS       = 20
LR           = 1e-4
WEIGHT_DECAY = 1e-4
GRAD_CLIP    = 1.0
SPLIT_TRAIN  = 0.70
SPLIT_VAL    = 0.15

# ─── Loss ────────────────────────────────────────────────────────────────────
FOCAL_GAMMA  = 2.0
FOCAL_ALPHA  = 0.85
DICE_WEIGHT  = 0.5

# ─── Parallel preprocessing ──────────────────────────────────────────────────
NUM_WORKERS_CACHE = min(8, os.cpu_count() or 4)   # Worker processes for cache build

total_viirs = VIIRS_CHANNELS + NIGHT_CHANNELS if USE_NIGHT else VIIRS_CHANNELS

print(" Configuration loaded.")
print(f"   VIIRS_Day channels      : {VIIRS_CHANNELS}")
print(f"   VIIRS_Night channels    : {NIGHT_CHANNELS if USE_NIGHT else 0}")
print(f"   Total VIIRS channels    : {total_viirs}")
print(f"   Input sequence length   : {SEQ_LEN} days")
print(f"   Prediction horizon      : {PRED_HORIZON} days (Days 4–{3+PRED_HORIZON})")
print(f"   ESRI_LULC               : EXCLUDED from model input")
print(f"   Spatial resolution      : {SPATIAL_SIZE}×{SPATIAL_SIZE}")
print(f"   Cache workers           : {NUM_WORKERS_CACHE}")


## 3. Dataset Diagnostics

In [ ]:
from collections import defaultdict, Counter

all_event_dirs = sorted([
    os.path.join(DATA_ROOT, d)
    for d in os.listdir(DATA_ROOT)
    if os.path.isdir(os.path.join(DATA_ROOT, d))
])

print(f"Total event directories found: {len(all_event_dirs)}")
print("─" * 60)

sample = all_event_dirs[0]
print(f"Sample event: {os.path.basename(sample)}")
for subdir in sorted(os.listdir(sample)):
    subdir_path = os.path.join(sample, subdir)
    if os.path.isdir(subdir_path):
        files = sorted(os.listdir(subdir_path))
        used = "(EXCLUDED — not used in model)" if "ESRI_LULC" in subdir else ""
        print(f"  [{subdir}]  ({len(files)} files) {used}")
        for f in files[:3]:
            print(f"    • {f}")

min_days_needed = SEQ_LEN + PRED_HORIZON
print(f"\nMinimum days required per event: {min_days_needed} (SEQ_LEN={SEQ_LEN} + PRED_HORIZON={PRED_HORIZON})")

valid_count = 0
skip_count  = 0
for d in all_event_dirs:
    day_f  = sorted(glob.glob(os.path.join(d, "VIIRS_Day",  "*.tif")))
    fire_f = sorted(glob.glob(os.path.join(d, "FirePred",   "*.tif")))
    n = min(len(day_f), len(fire_f))
    if n >= min_days_needed:
        valid_count += 1
    else:
        skip_count += 1

print(f"Events with enough data : {valid_count}")
print(f"Events skipped          : {skip_count}")

## 4. Parallel Preprocessing Cache
Builds `.pt` files in parallel using all available CPU cores. Safe to re-run — skips already-cached events.

In [ ]:
import multiprocessing as mp
from functools import partial

CACHE_DIR = os.path.join(SAVE_DIR, "cache")
os.makedirs(CACHE_DIR, exist_ok=True)

# ── Always rebuild all_event_dirs from current DATA_ROOT ─────────────────────
all_event_dirs = sorted([
    os.path.join(DATA_ROOT, d)
    for d in os.listdir(DATA_ROOT)
    if os.path.isdir(os.path.join(DATA_ROOT, d))
])
print(f" DATA_ROOT        : {DATA_ROOT}")
print(f" Event dirs found : {len(all_event_dirs)}")

# ── Low-level helpers (must be module-level for pickling in multiprocessing) ──
def _read_tif_all(path):
    with rasterio.open(path) as src:
        arr = src.read().astype(np.float32)
    return np.nan_to_num(arr, nan=0.0)

def _read_tif_band(path, band=1):
    with rasterio.open(path) as src:
        arr = src.read(band).astype(np.float32)
    return np.nan_to_num(arr, nan=0.0)

def _normalise_viirs(arr):
    out = np.zeros_like(arr, dtype=np.float32)
    for i in range(arr.shape[0]):
        b = arr[i]
        bmin, bmax = b.min(), b.max()
        if bmax - bmin > 1e-6:
            out[i] = (b - bmin) / (bmax - bmin)
    return out

def _normalise_frp(arr):
    arr = np.clip(arr, 0.0, None)
    pos = arr[arr > 0]
    if len(pos) == 0:
        return arr
    p99 = max(float(np.percentile(pos, 99)), 1.0)
    return np.clip(arr / p99, 0.0, 1.0)

def _fix_channels(arr, target_c):
    c = arr.shape[0]
    if c == target_c: return arr
    if c > target_c:  return arr[:target_c]
    pad = np.zeros((target_c - c, arr.shape[1], arr.shape[2]), dtype=arr.dtype)
    return np.concatenate([arr, pad], axis=0)

def _resize(arr, size):
    t = torch.tensor(arr, dtype=torch.float32).unsqueeze(0)
    t = F.interpolate(t, size=size, mode="bilinear", align_corners=False)
    return t.squeeze(0)

def preprocess_event(event_path, cache_dir, spatial_size, seq_len, pred_horizon,
                     viirs_channels, night_channels, use_night):
    """
    Preprocess one event directory.
    Produces one .pt file per valid sliding window.
    Returns list of saved file paths.
    """
    import os, glob, numpy as np, torch
    import torch.nn.functional as F
    import rasterio

    S = (spatial_size, spatial_size)
    min_days = seq_len + pred_horizon

    day_files   = sorted(glob.glob(os.path.join(event_path, "VIIRS_Day",   "*.tif")))
    fire_files  = sorted(glob.glob(os.path.join(event_path, "FirePred",    "*.tif")))
    night_files = sorted(glob.glob(os.path.join(event_path, "VIIRS_Night", "*.tif")))
    # ── ESRI_LULC is intentionally NOT loaded ────────────────────────────────

    n_avail   = min(len(day_files), len(fire_files))
    n_windows = n_avail - min_days + 1

    if n_windows < 1:
        return []

    event_name = os.path.basename(event_path)
    saved = []

    for start in range(n_windows):
        cache_key  = f"{event_name}_s{start:03d}.pt"
        cache_path = os.path.join(cache_dir, cache_key)
        if os.path.exists(cache_path):
            saved.append(cache_path)
            continue

        try:
            input_days   = day_files[start : start + seq_len]
            target_fires = fire_files[start + seq_len : start + seq_len + pred_horizon]

            # ── VIIRS day frames ──────────────────────────────────────────────
            viirs_frames = []
            for t_idx, f in enumerate(input_days):
                bands = _normalise_viirs(_read_tif_all(f))
                bands = _fix_channels(bands, viirs_channels)
                frame = _resize(bands, S)
                if use_night:
                    ni = start + t_idx
                    if ni < len(night_files):
                        n_arr = _normalise_viirs(_read_tif_all(night_files[ni]))
                        n_arr = _fix_channels(n_arr, night_channels)
                        n_t   = _resize(n_arr, S)
                    else:
                        n_t = torch.zeros(night_channels, *S)
                    frame = torch.cat([frame, n_t], dim=0)
                viirs_frames.append(frame)

            viirs_t = torch.stack(viirs_frames, dim=0)   # [T, C, S, S]

            # ── Weather: spatial mean of last input day ───────────────────────
            weather = viirs_t[-1].mean(dim=(1, 2))        # [C]

            # ── Multi-day targets ─────────────────────────────────────────────
            targets = []
            for tf in target_fires:
                tgt_raw = _read_tif_band(tf, band=1)
                tgt_raw = _normalise_frp(tgt_raw)
                tgt_t   = torch.tensor(tgt_raw, dtype=torch.float32).unsqueeze(0).unsqueeze(0)
                tgt_t   = F.interpolate(tgt_t, size=S, mode="bilinear",
                                        align_corners=False).squeeze(0)  # [1, S, S]
                targets.append(tgt_t)
            target_t = torch.stack(targets, dim=0)        # [P, 1, S, S]

            torch.save({
                "viirs"  : viirs_t,
                "weather": weather,
                "target" : target_t,
            }, cache_path)
            saved.append(cache_path)

        except Exception:
            pass

    return saved


# ── Main cache-build loop (parallel) ─────────────────────────────────────────
already = len(glob.glob(os.path.join(CACHE_DIR, "*.pt")))
print(f"Already cached  : {already} samples")

worker_fn = partial(
    preprocess_event,
    cache_dir      = CACHE_DIR,
    spatial_size   = SPATIAL_SIZE,
    seq_len        = SEQ_LEN,
    pred_horizon   = PRED_HORIZON,
    viirs_channels = VIIRS_CHANNELS,
    night_channels = NIGHT_CHANNELS,
    use_night      = USE_NIGHT,
)

t0 = time.time()
ctx = mp.get_context("fork")
with ctx.Pool(processes=NUM_WORKERS_CACHE) as pool:
    results = list(tqdm(
        pool.imap_unordered(worker_fn, all_event_dirs, chunksize=4),
        total=len(all_event_dirs),
        desc="Preprocessing events",
    ))

all_cache_files = sorted(glob.glob(os.path.join(CACHE_DIR, "*.pt")))
elapsed = time.time() - t0
print(f"\n Cache built in {elapsed:.1f}s")
print(f"   Total cached samples : {len(all_cache_files)}")

In [ ]:
# Debug: inspect what was found
print(f"DATA_ROOT: {DATA_ROOT}")
print(f"all_event_dirs count: {len(all_event_dirs)}")
print(f"First entry: {all_event_dirs[0]}")
print()

# Check its contents
d = all_event_dirs[0]
for sub in sorted(os.listdir(d)):
    sub_path = os.path.join(d, sub)
    if os.path.isdir(sub_path):
        files = os.listdir(sub_path)
        print(f"  [{sub}]  {len(files)} files")
    else:
        print(f"  {sub}  (file)")

## 5. Cached Dataset & DataLoaders

In [ ]:
class CachedFireDataset(Dataset):
    """
    Loads pre-processed .pt files.
    Each sample: viirs [T,C,S,S], weather [C], target [P,1,S,S]
    """
    def __init__(self, cache_files):
        self.files = cache_files

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        s = torch.load(self.files[idx], weights_only=True)
        return s["viirs"], s["weather"], s["target"]


# ── Split ─────────────────────────────────────────────────────────────────────
n_total = len(all_cache_files)
n_train = int(SPLIT_TRAIN * n_total)
n_val   = int(SPLIT_VAL   * n_total)
n_test  = n_total - n_train - n_val

generator = torch.Generator().manual_seed(SEED)
idx_splits = random_split(range(n_total), [n_train, n_val, n_test], generator=generator)

train_files = [all_cache_files[i] for i in idx_splits[0]]
val_files   = [all_cache_files[i] for i in idx_splits[1]]
test_files  = [all_cache_files[i] for i in idx_splits[2]]

train_ds = CachedFireDataset(train_files)
val_ds   = CachedFireDataset(val_files)
test_ds  = CachedFireDataset(test_files)

# Workers: 4 per GPU for fast loading; pin_memory for GPU transfer speed
loader_kw = dict(
    num_workers = min(4 * max(N_GPUS, 1), 8),
    pin_memory  = torch.cuda.is_available(),
    persistent_workers = True,
)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE * max(N_GPUS, 1),
                          shuffle=True,  **loader_kw)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE * max(N_GPUS, 1),
                          shuffle=False, **loader_kw)
test_loader  = DataLoader(test_ds,  batch_size=1, shuffle=False,
                          num_workers=2, pin_memory=torch.cuda.is_available())

print(f"Total cached samples : {n_total}")
print(f"   Train      : {len(train_ds)}")
print(f"   Validation : {len(val_ds)}")
print(f"   Test       : {len(test_ds)}")

# ── Shape check ───────────────────────────────────────────────────────────────
viirs_b, weather_b, tgt_b = next(iter(train_loader))
C_VIIRS = viirs_b.shape[2]

print(f"\nBatch shapes:")
print(f"  viirs   : {tuple(viirs_b.shape)}")
print(f"  weather : {tuple(weather_b.shape)}")
print(f"  target  : {tuple(tgt_b.shape)}")
print(f"\n  C_VIIRS = {C_VIIRS}  (used to build model)")


## 6. Model Architecture
**Changes from v1:**
- `DeepFireForecaster` now outputs `[B, PRED_HORIZON, 1, H, W]`
- A `WeatherFusion` module projects current-day weather (spatial mean of last VIIRS frame) into the decoder bottleneck
- ESRI_LULC is completely removed — no embedding layer
- Multi-head output: one UNet decoder head per prediction day (shared backbone, separate heads)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Building Blocks
# ═══════════════════════════════════════════════════════════════════════════════

class ConvBNReLU(nn.Module):
    def __init__(self, in_ch, out_ch, kernel=3, stride=1, padding=1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel, stride=stride, padding=padding, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.block(x)


class ResidualCNNBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            ConvBNReLU(in_ch, out_ch),
            ConvBNReLU(out_ch, out_ch),
        )
        self.skip = (nn.Conv2d(in_ch, out_ch, 1, bias=False)
                     if in_ch != out_ch else nn.Identity())
    def forward(self, x):
        return F.relu(self.conv(x) + self.skip(x), inplace=True)


# ═══════════════════════════════════════════════════════════════════════════════
# Spatial CNN Encoder  (no LULC input — ESRI_LULC excluded)
# ═══════════════════════════════════════════════════════════════════════════════

class SpatialEncoder(nn.Module):
    """
    Input  : [B*T, C_viirs, H, W]
    Outputs: s1 [BT, 32, H/2, W/2]
             s2 [BT, 64, H/4, W/4]
             s3 [BT, D,  H/8, W/8]
             bt [BT, D,  H/16,W/16]
    """
    def __init__(self, viirs_channels: int, embed_dim: int):
        super().__init__()
        self.enc1  = ResidualCNNBlock(viirs_channels, 32)
        self.pool1 = nn.MaxPool2d(2)
        self.enc2  = ResidualCNNBlock(32, 64)
        self.pool2 = nn.MaxPool2d(2)
        self.enc3  = ResidualCNNBlock(64, embed_dim)
        self.pool3 = nn.MaxPool2d(2)

    def forward(self, x):
        s1 = self.enc1(x)
        s2 = self.enc2(self.pool1(s1))
        s3 = self.enc3(self.pool2(s2))
        bt = self.pool3(s3)
        return s1, s2, s3, bt


# ═══════════════════════════════════════════════════════════════════════════════
# Weather Fusion  — conditions the decoder on current-day weather signal
# ═══════════════════════════════════════════════════════════════════════════════

class WeatherFusion(nn.Module):
    """
    Projects weather vector [B, C_weather] to spatial feature bias [B, D, 1, 1]
    which is broadcast-added to the bottleneck before decoding.
    """
    def __init__(self, weather_in: int, embed_dim: int, weather_dim: int = WEATHER_DIM):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Linear(weather_in, weather_dim),
            nn.ReLU(inplace=True),
            nn.Linear(weather_dim, embed_dim),
        )

    def forward(self, bottleneck, weather):
        """
        bottleneck : [B, D, Hb, Wb]
        weather    : [B, C_weather]  (spatial-mean of last VIIRS day frame)
        """
        bias = self.proj(weather)               # [B, D]
        bias = bias.unsqueeze(-1).unsqueeze(-1) # [B, D, 1, 1]
        return bottleneck + bias                # broadcast over spatial dims


# ═══════════════════════════════════════════════════════════════════════════════
# Spatiotemporal Transformer
# ═══════════════════════════════════════════════════════════════════════════════

class SpatiotemporalTransformer(nn.Module):
    def __init__(self, embed_dim, num_heads, num_layers, seq_len,
                 spatial_size_after_enc, dropout):
        super().__init__()
        Hf = Wf = spatial_size_after_enc // 8   # after 4× pooling
        self.Hf, self.Wf = Hf, Wf
        n_spatial = Hf * Wf
        self.spatial_pos  = nn.Parameter(torch.randn(1, 1, n_spatial, embed_dim) * 0.02)
        self.temporal_pos = nn.Parameter(torch.randn(1, seq_len, 1, embed_dim)   * 0.02)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=num_heads,
            dim_feedforward=embed_dim * 4,
            dropout=dropout, activation="gelu",
            batch_first=True, norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.norm        = nn.LayerNorm(embed_dim)

    def forward(self, x):
        B, T, D, H, W = x.shape
        x = x.view(B, T, D, H * W).permute(0, 1, 3, 2)
        x = x + self.spatial_pos + self.temporal_pos
        x = x.reshape(B, T * H * W, D)
        x = self.norm(self.transformer(x))
        x = x.view(B, T, H * W, D).permute(0, 1, 3, 2).contiguous()
        return x.view(B, T, D, H, W)


# ═══════════════════════════════════════════════════════════════════════════════
# Temporal Attention Pooling
# ═══════════════════════════════════════════════════════════════════════════════

class TemporalAttentionPooling(nn.Module):
    def __init__(self, embed_dim):
        super().__init__()
        self.score_net = nn.Sequential(
            nn.Conv2d(embed_dim, embed_dim // 2, 1),
            nn.ReLU(inplace=True),
            nn.Conv2d(embed_dim // 2, 1, 1),
        )

    def forward(self, x):
        B, T, D, H, W = x.shape
        scores  = self.score_net(x.view(B * T, D, H, W))   # [BT, 1, H, W]
        scores  = scores.view(B, T, H * W).mean(dim=2)      # [B, T]
        weights = F.softmax(scores, dim=1)
        w       = weights.view(B, T, 1, 1, 1)
        pooled  = (x * w).sum(dim=1)                        # [B, D, H, W]
        return pooled, weights


# ═══════════════════════════════════════════════════════════════════════════════
# UNet Decoder  (one instance per prediction day)
# ═══════════════════════════════════════════════════════════════════════════════

class UNetDecoder(nn.Module):
    def __init__(self, embed_dim: int = EMBED_DIM):
        super().__init__()
        self.up3    = nn.Sequential(nn.ConvTranspose2d(embed_dim, 64, 4, 2, 1),
                                    nn.BatchNorm2d(64), nn.ReLU(True))
        self.ref3   = ResidualCNNBlock(64 + embed_dim, 64)
        self.up2    = nn.Sequential(nn.ConvTranspose2d(64, 32, 4, 2, 1),
                                    nn.BatchNorm2d(32), nn.ReLU(True))
        self.ref2   = ResidualCNNBlock(32 + 64, 32)
        self.up1    = nn.Sequential(nn.ConvTranspose2d(32, 16, 4, 2, 1),
                                    nn.BatchNorm2d(16), nn.ReLU(True))
        self.ref1   = ResidualCNNBlock(16 + 32, 16)
        self.head   = nn.Conv2d(16, 1, 1)

    def forward(self, bottleneck, s3, s2, s1):
        x = self.ref3(torch.cat([self.up3(bottleneck), s3], dim=1))
        x = self.ref2(torch.cat([self.up2(x),          s2], dim=1))
        x = self.ref1(torch.cat([self.up1(x),          s1], dim=1))
        return torch.sigmoid(self.head(x))     # [B, 1, H, W]


# ═══════════════════════════════════════════════════════════════════════════════
# Full DeepFire Forecaster v2
# ═══════════════════════════════════════════════════════════════════════════════

class DeepFireForecaster(nn.Module):
    """
    Inputs:
        viirs   : [B, T, C, H, W]   — T days of VIIRS day+night imagery
        weather : [B, C]             — spatial-mean of last VIIRS day (current conditions)
    Output:
        preds        : [B, P, 1, H, W]   — P-day fire probability maps
        attn_weights : [B, T]             — temporal attention weights
    """

    def __init__(
        self,
        viirs_channels : int   = None,     # set from data
        embed_dim      : int   = EMBED_DIM,
        num_heads      : int   = NUM_HEADS,
        num_layers     : int   = NUM_LAYERS,
        seq_len        : int   = SEQ_LEN,
        pred_horizon   : int   = PRED_HORIZON,
        spatial_size   : int   = SPATIAL_SIZE,
        dropout        : float = DROPOUT,
        weather_dim    : int   = WEATHER_DIM,
    ):
        super().__init__()
        if viirs_channels is None:
            viirs_channels = VIIRS_CHANNELS + NIGHT_CHANNELS if USE_NIGHT else VIIRS_CHANNELS
        self.pred_horizon = pred_horizon

        self.encoder     = SpatialEncoder(viirs_channels, embed_dim)
        self.transformer = SpatiotemporalTransformer(
            embed_dim, num_heads, num_layers, seq_len, spatial_size, dropout)
        self.temp_pool   = TemporalAttentionPooling(embed_dim)
        self.weather_fuse = WeatherFusion(viirs_channels, embed_dim, weather_dim)

        # One decoder head per prediction day (shared backbone, separate heads)
        self.decoders = nn.ModuleList([
            UNetDecoder(embed_dim) for _ in range(pred_horizon)
        ])

    def forward(self, viirs, weather):
        B, T, C, H, W = viirs.shape

        # ── Spatial encoding (shared per-frame weights) ───────────────────────
        viirs_flat = viirs.view(B * T, C, H, W)
        s1, s2, s3, bottleneck = self.encoder(viirs_flat)

        def _rbatch(t, ch, h, w):
            return t.view(B, T, ch, h, w)

        _, _, H3, W3 = s3.shape
        _, _, H2, W2 = s2.shape
        _, _, H1, W1 = s1.shape
        _, D,  Hb, Wb = bottleneck.shape

        bt_seq = _rbatch(bottleneck, D, Hb, Wb)          # [B, T, D, Hb, Wb]

        # ── Spatiotemporal transformer ────────────────────────────────────────
        bt_seq = self.transformer(bt_seq)

        # ── Temporal pooling ──────────────────────────────────────────────────
        pooled, attn_weights = self.temp_pool(bt_seq)    # [B, D, Hb, Wb]

        # ── Weather fusion (current-day conditions → decoder bias) ────────────
        pooled = self.weather_fuse(pooled, weather)

        # ── Aggregate skip connections across T ───────────────────────────────
        s3_agg = _rbatch(s3, s3.shape[1], H3, W3).mean(dim=1)
        s2_agg = _rbatch(s2, s2.shape[1], H2, W2).mean(dim=1)
        s1_agg = _rbatch(s1, s1.shape[1], H1, W1).mean(dim=1)

        # ── Multi-day decoding ────────────────────────────────────────────────
        preds = torch.stack([
            dec(pooled, s3_agg, s2_agg, s1_agg)
            for dec in self.decoders
        ], dim=1)                                         # [B, P, 1, H, W]

        return preds, attn_weights


# ── Instantiate ───────────────────────────────────────────────────────────────
model = DeepFireForecaster(viirs_channels=C_VIIRS).to(DEVICE)

# Wrap in DataParallel if multiple GPUs available (T4×2)
if N_GPUS > 1:
    model = nn.DataParallel(model)
    print(f" DataParallel enabled across {N_GPUS} GPUs")

total_params = sum(p.numel() for p in model.parameters())
trainable    = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f" DeepFireForecaster v2 initialised")
print(f"   Total parameters     : {total_params:,}")
print(f"   Trainable parameters : {trainable:,}")

# ── Dry-run ───────────────────────────────────────────────────────────────────
with torch.no_grad():
    dv = torch.randn(2, SEQ_LEN, C_VIIRS, SPATIAL_SIZE, SPATIAL_SIZE).to(DEVICE)
    dw = torch.randn(2, C_VIIRS).to(DEVICE)
    pred_out, attn_out = model(dv, dw)
    print(f"   Dry-run pred shape   : {tuple(pred_out.shape)}  ")
    print(f"   Attn weights shape   : {tuple(attn_out.shape)}  ")


## 7. Multi-Day Masked Focal-Dice Loss

In [ ]:
class MaskedFocalDiceLoss(nn.Module):
    """
    Multi-day Focal-Dice loss.
    pred   : [B, P, 1, H, W]
    target : [B, P, 1, H, W]
    Loss is averaged over all P prediction days.
    """
    def __init__(self, gamma=FOCAL_GAMMA, alpha=FOCAL_ALPHA,
                 dice_weight=DICE_WEIGHT, eps=1e-6):
        super().__init__()
        self.gamma, self.alpha, self.dice_weight, self.eps = gamma, alpha, dice_weight, eps

    def _one_day(self, pred, target):
        pred   = pred.float()
        target = target.float()
        mask = (~torch.isnan(target)) & (target >= 0.0) & (target <= 1.0)
        mask = mask.float()
        p = pred * mask
        t = target * mask
        n = mask.sum().clamp(min=1.0)
        bce   = F.binary_cross_entropy(p, t, reduction="none")
        pt    = torch.exp(-bce)
        focal = self.alpha * (1.0 - pt) ** self.gamma * bce
        focal_loss = (focal * mask).sum() / n
        pf = p.reshape(-1); tf = t.reshape(-1)
        dice_loss = 1.0 - (2.0 * (pf * tf).sum() + self.eps) / (
            pf.sum() + tf.sum() + self.eps)
        return (1.0 - self.dice_weight) * focal_loss + self.dice_weight * dice_loss

    def forward(self, pred, target):
        P = pred.shape[1]
        return sum(self._one_day(pred[:, p], target[:, p]) for p in range(P)) / P


criterion = MaskedFocalDiceLoss()
print(" MaskedFocalDiceLoss (multi-day) ready.")

with torch.no_grad():
    dp = torch.rand(2, PRED_HORIZON, 1, 64, 64)
    dt = (torch.rand(2, PRED_HORIZON, 1, 64, 64) > 0.9).float()
    lv = criterion(dp, dt)
    print(f"   Sanity-check loss   : {lv.item():.6f}  ")

## 8. Training Loop (AMP + DataParallel)

In [ ]:
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
scaler    = torch.cuda.amp.GradScaler(enabled=USE_AMP)

def binary_iou(pred, target, threshold=0.5):
    p = (pred  >= threshold).float()
    t = (target >= threshold).float()
    intersection = (p * t).sum()
    union        = (p + t).clamp(0, 1).sum()
    return (intersection / (union + 1e-6)).item()

history = {"train_loss": [], "val_loss": [], "val_iou": [], "lr": []}
best_val_loss = float("inf")

print(f"{'='*62}")
print(f"  Training DeepFireForecaster v2  ({EPOCHS} epochs, {DEVICE})")
print(f"  Predicting {PRED_HORIZON} future days simultaneously")
print(f"{'='*62}\n")

for epoch in range(1, EPOCHS + 1):
    # ─── TRAIN ───────────────────────────────────────────────────────────────
    model.train()
    train_loss = 0.0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch:02d}/{EPOCHS} [Train]", leave=False)
    for viirs_b, weather_b, tgt_b in pbar:
        viirs_b   = viirs_b.to(DEVICE, non_blocking=True)
        weather_b = weather_b.to(DEVICE, non_blocking=True)
        tgt_b     = tgt_b.to(DEVICE, non_blocking=True)

        optimizer.zero_grad()

        # AMP only for the forward pass — loss computed in float32
        with torch.cuda.amp.autocast(enabled=USE_AMP):
            pred, _ = model(viirs_b, weather_b)

        loss = criterion(pred, tgt_b)   # always float32, outside autocast

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item()
        pbar.set_postfix({"loss": f"{loss.item():.5f}"})

    avg_train = train_loss / len(train_loader)

    # ─── VALIDATE ────────────────────────────────────────────────────────────
    model.eval()
    val_loss = val_iou = 0.0
    with torch.no_grad():
        for viirs_b, weather_b, tgt_b in val_loader:
            viirs_b   = viirs_b.to(DEVICE)
            weather_b = weather_b.to(DEVICE)
            tgt_b     = tgt_b.to(DEVICE)

            with torch.cuda.amp.autocast(enabled=USE_AMP):
                pred, _ = model(viirs_b, weather_b)

            val_loss += criterion(pred, tgt_b).item()   # float32, outside autocast
            val_iou  += binary_iou(pred.cpu(), tgt_b.cpu())

    avg_val = val_loss / len(val_loader)
    avg_iou = val_iou  / len(val_loader)
    cur_lr  = optimizer.param_groups[0]["lr"]
    scheduler.step()

    history["train_loss"].append(avg_train)
    history["val_loss"].append(avg_val)
    history["val_iou"].append(avg_iou)
    history["lr"].append(cur_lr)

    improved = avg_val < best_val_loss
    if improved:
        best_val_loss = avg_val
        state = model.module.state_dict() if N_GPUS > 1 else model.state_dict()
        torch.save({"epoch": epoch, "model": state,
                    "optimizer": optimizer.state_dict(),
                    "val_loss": avg_val, "val_iou": avg_iou,
                    "history": history}, MODEL_PATH)

    marker = " ★ saved" if improved else ""
    print(f"Epoch {epoch:02d}/{EPOCHS}  "
          f"train={avg_train:.5f}  val={avg_val:.5f}  "
          f"IoU={avg_iou:.4f}  lr={cur_lr:.2e}{marker}")

print(f"\n Training complete. Best val loss: {best_val_loss:.5f}")

## 9. Training Diagnostics

In [ ]:
EVAL_DIR   = os.path.join(SAVE_DIR, "evaluation")
PLOTS_DIR  = os.path.join(EVAL_DIR, "plots")
STATS_DIR  = os.path.join(EVAL_DIR, "stats")
for d in [EVAL_DIR, PLOTS_DIR, STATS_DIR]:
    os.makedirs(d, exist_ok=True)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].plot(history["train_loss"], label="Train"); axes[0].plot(history["val_loss"], label="Val")
axes[0].set_title("Loss"); axes[0].legend(); axes[0].set_xlabel("Epoch")
axes[1].plot(history["val_iou"], color="green")
axes[1].set_title("Validation IoU (fire class)"); axes[1].set_xlabel("Epoch")
axes[2].plot(history["lr"], color="orange")
axes[2].set_title("Learning Rate"); axes[2].set_xlabel("Epoch")
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "learning_curves.png"), dpi=150, bbox_inches="tight")
plt.show(); plt.close()
print(" Learning curves saved.")


## 10. Evaluation & Result Plots

**Format:** For each category (3 best / 3 worst / 3 random), one composite figure per sample containing:
- Row 1: Input Day 1, Day 2, Day 3 thermal band
- Row 2: Prediction Day 4, Ground Truth Day 4, Difference Map
- Row 3 (if PRED_HORIZON ≥ 2): Prediction Day 5, Ground Truth Day 5, Difference Map
- Row 4 (if PRED_HORIZON ≥ 3): Prediction Day 6, Ground Truth Day 6, Difference Map

→ Each figure: **12 images** (6 columns × 2+ rows). 3 figures per category = **9 total figures**.

In [ ]:
import csv, random as pyrandom

# ── Load best checkpoint ──────────────────────────────────────────────────────
checkpoint = torch.load(MODEL_PATH, map_location=DEVICE)
_state = checkpoint["model"]
if N_GPUS > 1:
    model.module.load_state_dict(_state)
else:
    model.load_state_dict(_state)
model.eval()
print(f"Loaded checkpoint (epoch {checkpoint['epoch']}, val_loss={checkpoint['val_loss']:.5f})")

THRESHOLD = 0.4

def compute_metrics(pred_prob, target, thresh=THRESHOLD):
    pred_bin = (pred_prob >= thresh).astype(np.float32)
    tgt_bin  = (target    >= thresh).astype(np.float32)
    tp = float((pred_bin * tgt_bin).sum())
    fp = float((pred_bin * (1 - tgt_bin)).sum())
    fn = float(((1 - pred_bin) * tgt_bin).sum())
    tn = float(((1 - pred_bin) * (1 - tgt_bin)).sum())
    n  = pred_prob.size; eps = 1e-7
    precision = tp / (tp + fp + eps)
    recall    = tp / (tp + fn + eps)
    f1        = 2 * precision * recall / (precision + recall + eps)
    iou       = tp / (tp + fp + fn + eps)
    accuracy  = (tp + tn) / n
    mae       = float(np.abs(pred_prob - target).mean())
    rmse      = float(np.sqrt(((pred_prob - target) ** 2).mean()))
    return dict(TP=tp, FP=fp, FN=fn, TN=tn,
                Precision=precision, Recall=recall,
                F1=f1, IoU=iou, Accuracy=accuracy,
                MAE=mae, RMSE=rmse)

# ── Run inference on all test samples ────────────────────────────────────────
print(f"\nRunning inference on {len(test_ds)} test samples …")
all_results = []   # list of dicts with numpy arrays + metrics

with torch.no_grad():
    for idx in tqdm(range(len(test_ds)), desc="Evaluating"):
        viirs_b, weather_b, tgt_b = test_ds[idx]

        preds_t, attn = model(
            viirs_b.unsqueeze(0).to(DEVICE),
            weather_b.unsqueeze(0).to(DEVICE),
        )
        preds_np = preds_t.cpu().squeeze(0).numpy()   # [P, 1, H, W]
        tgt_np   = tgt_b.numpy()                      # [P, 1, H, W]
        viirs_np = viirs_b.numpy()                    # [T, C, H, W]

        # Aggregate IoU across all prediction days
        day_metrics = []
        for p in range(PRED_HORIZON):
            m = compute_metrics(preds_np[p, 0], tgt_np[p, 0])
            day_metrics.append(m)
        mean_iou = float(np.mean([m["IoU"] for m in day_metrics]))

        all_results.append({
            "idx"        : idx,
            "viirs_np"   : viirs_np,
            "preds_np"   : preds_np,
            "tgt_np"     : tgt_np,
            "day_metrics": day_metrics,
            "mean_iou"   : mean_iou,
        })

# ── Sort and select samples ───────────────────────────────────────────────────
sorted_by_iou = sorted(all_results, key=lambda r: r["mean_iou"])
best_3    = sorted_by_iou[-3:][::-1]          # highest IoU
worst_3   = sorted_by_iou[:3]                 # lowest IoU
used_idxs = {r["idx"] for r in best_3 + worst_3}
remaining = [r for r in all_results if r["idx"] not in used_idxs]
random_3  = pyrandom.sample(remaining, min(3, len(remaining)))

def make_composite_plot(result, category, rank, save_dir):
    """
    Creates a composite figure: 12 subplots arranged as:
      Row 1: [Day1 thermal] [Day2 thermal] [Day3 thermal]   ← inputs
      Row 2: [Pred Day4] [Truth Day4] [Diff Day4]
      Row 3: [Pred Day5] [Truth Day5] [Diff Day5]            (if PRED_HORIZON>=2)
      Row 4: [Pred Day6] [Truth Day6] [Diff Day6]            (if PRED_HORIZON>=3)
    Total: (1 + PRED_HORIZON) rows × 3 cols = (4 × 3 = 12) images for PRED_HORIZON=3
    """
    viirs_np  = result["viirs_np"]
    preds_np  = result["preds_np"]
    tgt_np    = result["tgt_np"]
    metrics   = result["day_metrics"]
    mean_iou  = result["mean_iou"]
    idx       = result["idx"]

    n_rows = 1 + PRED_HORIZON    # 4 rows for PRED_HORIZON=3
    fig, axes = plt.subplots(n_rows, 3, figsize=(15, 5 * n_rows))

    cmap_therm = plt.cm.viridis
    cmap_fire  = plt.cm.inferno
    cmap_diff  = plt.cm.RdBu_r

    # ── Row 0: Input VIIRS thermal (band 0) ──────────────────────────────────
    for day in range(3):
        ax = axes[0, day]
        ax.imshow(viirs_np[day, 0], cmap=cmap_therm, vmin=0, vmax=1)
        ax.set_title(f"Input Day {day+1}\nVIIRS Thermal (Band 1)", fontsize=10)
        ax.axis("off")

    # ── Rows 1…P: Prediction / Ground-truth / Difference ─────────────────────
    for p in range(PRED_HORIZON):
        pred_p = preds_np[p, 0]
        tgt_p  = tgt_np[p, 0]
        diff_p = pred_p - tgt_p
        m      = metrics[p]

        ax_pred = axes[p + 1, 0]
        ax_tgt  = axes[p + 1, 1]
        ax_diff = axes[p + 1, 2]

        ax_pred.imshow(pred_p, cmap=cmap_fire, vmin=0, vmax=1)
        ax_pred.set_title(f"Predicted Day {p+4}\n"
                          f"IoU={m['IoU']:.3f}  F1={m['F1']:.3f}", fontsize=9)
        ax_pred.axis("off")

        ax_tgt.imshow(tgt_p, cmap=cmap_fire, vmin=0, vmax=1)
        ax_tgt.set_title(f"Ground Truth Day {p+4}\n"
                         f"Prec={m['Precision']:.3f}  Rec={m['Recall']:.3f}", fontsize=9)
        ax_tgt.axis("off")

        vmax_d = max(abs(diff_p).max(), 0.01)
        im_d = ax_diff.imshow(diff_p, cmap=cmap_diff, vmin=-vmax_d, vmax=vmax_d)
        ax_diff.set_title(f"Difference Map Day {p+4}\n"
                          f"MAE={m['MAE']:.4f}  RMSE={m['RMSE']:.4f}", fontsize=9)
        ax_diff.axis("off")
        fig.colorbar(im_d, ax=ax_diff, fraction=0.046, pad=0.04)

    fig.suptitle(
        f"{category} #{rank+1}  |  Sample {idx}  |  Mean IoU = {mean_iou:.3f}",
        fontsize=13, fontweight="bold", y=1.01
    )
    plt.tight_layout()

    fname = f"{category.lower().replace(' ', '_')}_{rank+1:02d}_sample{idx:04d}.png"
    fpath = os.path.join(save_dir, fname)
    plt.savefig(fpath, dpi=150, bbox_inches="tight")
    plt.close(fig)
    return fpath


# ── Generate all 9 composite figures ─────────────────────────────────────────
RESULTS_DIR = os.path.join(EVAL_DIR, "result_plots")
os.makedirs(RESULTS_DIR, exist_ok=True)

print("\nGenerating composite result plots …")
saved_plots = []

for cat_name, cat_samples in [("Best", best_3), ("Worst", worst_3), ("Random", random_3)]:
    for rank, result in enumerate(cat_samples):
        fp = make_composite_plot(result, cat_name, rank, RESULTS_DIR)
        saved_plots.append(fp)
        print(f"  Saved: {os.path.basename(fp)}  (mean_iou={result['mean_iou']:.3f})")

print(f"\n {len(saved_plots)} composite plots saved to {RESULTS_DIR}")

# ── Save test metrics CSV ──────────────────────────────────────────────────────
csv_path = os.path.join(STATS_DIR, "test_metrics.csv")
with open(csv_path, "w", newline="") as f:
    day_headers = [f"Day{d+4}_{k}"
                   for d in range(PRED_HORIZON)
                   for k in ["IoU", "F1", "Precision", "Recall", "MAE"]]
    writer = csv.DictWriter(f, fieldnames=["sample_idx", "mean_iou"] + day_headers)
    writer.writeheader()
    for r in all_results:
        row = {"sample_idx": r["idx"], "mean_iou": r["mean_iou"]}
        for d, m in enumerate(r["day_metrics"]):
            for k in ["IoU", "F1", "Precision", "Recall", "MAE"]:
                row[f"Day{d+4}_{k}"] = round(m[k], 5)
        writer.writerow(row)
print(f" Metrics CSV saved: {csv_path}")

# ── Print aggregate stats ──────────────────────────────────────────────────────
print("\n── Aggregate Test Statistics (mean over all samples) ─────────────────")
for d in range(PRED_HORIZON):
    ious = [r["day_metrics"][d]["IoU"] for r in all_results]
    f1s  = [r["day_metrics"][d]["F1"]  for r in all_results]
    print(f"  Day {d+4}: IoU={np.mean(ious):.4f} ± {np.std(ious):.4f}   "
          f"F1={np.mean(f1s):.4f} ± {np.std(f1s):.4f}")


## 11. Package All Outputs for Download

In [ ]:
import zipfile

ZIP_PATH = os.path.join(SAVE_DIR, "deepfire_v2_outputs.zip")

with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    # Model checkpoint
    if os.path.exists(MODEL_PATH):
        zf.write(MODEL_PATH, "deepfire_best.pt")

    # Learning curves
    for f in sorted(glob.glob(os.path.join(PLOTS_DIR, "*.png"))):
        zf.write(f, os.path.join("plots", os.path.basename(f)))

    # 9 composite result plots (best/worst/random)
    for f in sorted(glob.glob(os.path.join(RESULTS_DIR, "*.png"))):
        zf.write(f, os.path.join("result_plots", os.path.basename(f)))

    # Stats
    for f in sorted(glob.glob(os.path.join(STATS_DIR, "*"))):
        zf.write(f, os.path.join("stats", os.path.basename(f)))

zip_mb = os.path.getsize(ZIP_PATH) / 1e6
print(f" ZIP created: {ZIP_PATH}  ({zip_mb:.1f} MB)")
print(f"   → Go to Kaggle sidebar > Output tab to download")


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Model Statistics
# ═══════════════════════════════════════════════════════════════════════════════
import torch
from collections import OrderedDict

_model = model.module if N_GPUS > 1 else model

# ── Parameter counts ──────────────────────────────────────────────────────────
total_params     = sum(p.numel() for p in _model.parameters())
trainable_params = sum(p.numel() for p in _model.parameters() if p.requires_grad)
frozen_params    = total_params - trainable_params

# ── Per-module breakdown ──────────────────────────────────────────────────────
module_stats = OrderedDict()
for name, module in _model.named_children():
    n = sum(p.numel() for p in module.parameters())
    module_stats[name] = n

# ── Memory estimate ───────────────────────────────────────────────────────────
param_mb  = total_params * 4 / 1024**2          # float32 = 4 bytes
grad_mb   = trainable_params * 4 / 1024**2      # one gradient per trainable param
buffer_mb = sum(b.numel() * b.element_size()
                for b in _model.buffers()) / 1024**2

# ── GPU memory (if available) ─────────────────────────────────────────────────
if torch.cuda.is_available():
    gpu_allocated = torch.cuda.memory_allocated() / 1024**2
    gpu_reserved  = torch.cuda.memory_reserved()  / 1024**2
    gpu_total     = torch.cuda.get_device_properties(0).total_memory / 1024**2

# ── Input/output shapes ───────────────────────────────────────────────────────
dummy_viirs = torch.randn(1, SEQ_LEN, C_VIIRS, SPATIAL_SIZE, SPATIAL_SIZE).to(DEVICE)
dummy_weather = torch.randn(1, C_VIIRS).to(DEVICE)
with torch.no_grad():
    dummy_pred, dummy_attn = _model(dummy_viirs, dummy_weather)

# ── Print report ──────────────────────────────────────────────────────────────
print("=" * 62)
print("   DeepFire Forecaster v3 — Model Statistics")
print("=" * 62)

print("\n── Architecture ─────────────────────────────────────────────")
print(f"  Input  viirs   : {tuple(dummy_viirs.shape)}   [B, T, C, H, W]")
print(f"  Input  weather : {tuple(dummy_weather.shape)}            [B, C]")
print(f"  Output preds   : {tuple(dummy_pred.shape)}  [B, P, 1, H, W]")
print(f"  Output attn    : {tuple(dummy_attn.shape)}         [B, T]")
print(f"\n  SEQ_LEN        : {SEQ_LEN}   input days")
print(f"  PRED_HORIZON   : {PRED_HORIZON}   output days")
print(f"  SPATIAL_SIZE   : {SPATIAL_SIZE}×{SPATIAL_SIZE}")
print(f"  EMBED_DIM      : {EMBED_DIM}")
print(f"  NUM_HEADS      : {NUM_HEADS}")
print(f"  NUM_LAYERS     : {NUM_LAYERS}   transformer layers")
print(f"  C_VIIRS        : {C_VIIRS}  ({VIIRS_CHANNELS} day + {NIGHT_CHANNELS} night)")

print("\n── Parameters ───────────────────────────────────────────────")
print(f"  Total          : {total_params:>12,}")
print(f"  Trainable      : {trainable_params:>12,}")
print(f"  Frozen         : {frozen_params:>12,}")

print("\n── Per-module breakdown ──────────────────────────────────────")
for name, count in module_stats.items():
    pct = 100 * count / total_params
    bar = "█" * int(pct / 2)
    print(f"  {name:<20} {count:>10,}  ({pct:5.1f}%)  {bar}")

print("\n── Memory estimate ──────────────────────────────────────────")
print(f"  Parameters     : {param_mb:>8.1f} MB  (float32)")
print(f"  Gradients      : {grad_mb:>8.1f} MB  (float32)")
print(f"  Buffers        : {buffer_mb:>8.1f} MB")
print(f"  Model total    : {param_mb + grad_mb + buffer_mb:>8.1f} MB  (approx)")

if torch.cuda.is_available():
    print(f"\n── GPU Memory ({torch.cuda.get_device_name(0)}) ──────────────────")
    print(f"  Allocated      : {gpu_allocated:>8.1f} MB")
    print(f"  Reserved       : {gpu_reserved:>8.1f} MB")
    print(f"  Total VRAM     : {gpu_total:>8.1f} MB")
    print(f"  VRAM used      : {100*gpu_allocated/gpu_total:>7.1f} %")
    if N_GPUS > 1:
        for i in range(N_GPUS):
            alloc = torch.cuda.memory_allocated(i) / 1024**2
            res   = torch.cuda.memory_reserved(i)  / 1024**2
            print(f"  GPU {i} allocated : {alloc:>8.1f} MB  reserved: {res:.1f} MB")

print("\n── Training config ──────────────────────────────────────────")
print(f"  EPOCHS         : {EPOCHS}")
print(f"  BATCH_SIZE     : {BATCH_SIZE} × {max(N_GPUS,1)} GPU = {BATCH_SIZE*max(N_GPUS,1)} effective")
print(f"  LR             : {LR}")
print(f"  WEIGHT_DECAY   : {WEIGHT_DECAY}")
print(f"  GRAD_CLIP      : {GRAD_CLIP}")
print(f"  AMP            : {USE_AMP}")
print(f"  DataParallel   : {N_GPUS > 1} ({N_GPUS} GPUs)")
print(f"  Optimizer      : AdamW + CosineAnnealingLR")
print(f"  Loss           : Focal-Dice (γ={FOCAL_GAMMA}, α={FOCAL_ALPHA}, dice_w={DICE_WEIGHT})")

print("\n── Dataset ──────────────────────────────────────────────────")
print(f"  Total samples  : {len(train_ds) + len(val_ds) + len(test_ds):,}")
print(f"  Train          : {len(train_ds):,}  ({100*SPLIT_TRAIN:.0f}%)")
print(f"  Validation     : {len(val_ds):,}  ({100*SPLIT_VAL:.0f}%)")
print(f"  Test           : {len(test_ds):,}  ({100*(1-SPLIT_TRAIN-SPLIT_VAL):.0f}%)")
print(f"  ESRI_LULC      : excluded")
print("=" * 62)